# formal and informal stratigraphic name extraction with llm for dynamic prompting

In [18]:
from IPython.display import HTML
# TEXT = "The Sevy Dolomite was named in 1935 by T.B. Nolan in the Deep Creek Range of western Utah in western Juab County. Osmond (1954, 1962) described its extent throughout 100,000 square miles (259,000 sq kilometers) in Nevada, Utah, California and Idaho, where it apparently unconformably overlies Silurian dolomite and is overlain by middle Devonian dolomite (Hintze and Davis, 2003). The Sevy Dolomite is easily recognized in Millard County by its uniform light-gray color, regular bedding in beds up to approximately 2.0 feet (0.6 m) thick, and the fine-grained faintly laminated texture of the unit. In its uppermost part it includes scattered thin horizons of rusty-weathered, frosted quartz sand grains of wind-blown origin that commonly float as thin seams or individual grains in the dolomite matrix (Hintze and Davis, 2003)."
# TEXT = "The Simonson Dolomite is the lowermost lithologic unit intersected by drilling at the Pan property. This unit is not exposed on the surface. Thickness ranges from 500 to 1,300 ft thick in White Pine County (Smith, 1976) but only the top portion of the dolomite has been drilled at South Pan. The dolomite is a light gray, massively bedded unit.\nThe oldest lithologic unit exposed in the northern Pancake Range is the Late Devonian Devil’s Gate Limestone. This unit is massive to thinly bedded, medium to dark gray, fine to coarse grained limestone. Thickness of the unit ranges from about 1000 ft to 2500 ft locally. The Devil’s Gate Limestone is the secondary host of gold mineralization at the Pan property."
TEXT = "Grant Ranges, but at the latitude of upper Ellison Creek in the White Pine Range, we have mapped about 75 to 100 feet of Pilot. Farther north in the White Pine Range, Humphrey (1960) reports that there are 150 to 200 feet of this shale, and about 6 miles north of Mount Grafton, in the southern Schell Creek Range, we found that it was in some places as much as 50 feet thick but thinned out to a knife edge. Nolan and others (1956) report that the Pilot Shale in the Eureka area is 315 to 425 feet thick. Rigby (1960) found 400 to 500 feet of Pilot Shale south of Overland Pass in the southern Ruby Mountains. We have mapped Pilot Shale with an estimated thickness of 400 to 600 feet in the Ruby Mountains just north of the county line. The thickest sections of the Pilot are in the northeastern part of the county: Fritz (1960) found it to be 615 feet thick in the southern Cherry Creek Range, Dechert (1967) gives its thickness in the northern Schell Creek Range as 700 feet, and we have estimated that it is 950 feet thick in the Red Hills."


HTML('<p style="font-size: 18px">'+TEXT.replace(".", ".<br><br>")+'</p>')

In [1]:
from text2graph.dynamic_prompt import StratPromptHandlerV4


ph = StratPromptHandlerV4(model="mixtral")
llm_processed_result = ph.get_known_entities(TEXT)
llm_processed_result

['Pilot', 'Pilot Shale']

# Extraction with llm based dynamic prompting 

In [2]:
from text2graph.llm import ask_llm

triplets_result = await ask_llm(
    text=TEXT,
    prompt_handler=ph,
    model='mixtral',
    temperature=0.0,
    to_triplets=False,  # For debugging intermediate steps
    hydrate=False
)
print(triplets_result)

{
"triplets": [
{"location": "Grant Ranges", "relationship": "contains", "stratigraphic_name": "Pilot"},
{"location": "upper Ellison Creek, White Pine Range", "relationship": "includes", "stratigraphic_name": "Pilot"},
{"location": "north of Mount Grafton, southern Schell Creek Range", "relationship": "contains", "stratigraphic_name": "Pilot"},
{"location": "Eureka area", "relationship": "contains", "stratigraphic_name": "Pilot Shale"},
{"location": "south of Overland Pass, southern Ruby Mountains", "relationship": "contains", "stratigraphic_name": "Pilot Shale"},
{"location": "Ruby Mountains just north of the county line", "relationship": "contains", "stratigraphic_name": "Pilot Shale"},
{"location": "southern Cherry Creek Range", "relationship": "contains", "stratigraphic_name": "Pilot"},
{"location": "northern Schell Creek Range", "relationship": "contains", "stratigraphic_name": "Pilot Shale"},
{"location": "Red Hills", "relationship": "contains", "stratigraphic_name": "Pilot"}
]
}

In [20]:
from text2graph.llm import post_process

pp_result = await post_process(triplets_result, prompt_handler=ph, hydrate=False)
[(t["object"]["strat_name"], t["predicate"], t["subject"]["name"]) for t in pp_result.model_dump()["triplets"]]

[('Pilot Shale', 'contains', 'Grant Ranges'),
 ('Pilot Shale', 'includes', 'upper Ellison Creek, White Pine Range'),
 ('Pilot Shale',
  'contains',
  'north of Mount Grafton, southern Schell Creek Range'),
 ('Pilot Shale', 'contains', 'Eureka area'),
 ('Pilot Shale',
  'contains',
  'south of Overland Pass, southern Ruby Mountains'),
 ('Pilot Shale', 'contains', 'Ruby Mountains just north of the county line'),
 ('Pilot Shale', 'contains', 'southern Cherry Creek Range'),
 ('Pilot Shale', 'contains', 'northern Schell Creek Range'),
 ('Pilot Shale', 'contains', 'Red Hills')]